# Global summaries and tree ensembles

Using the [UCI German Credit][gc] data (see `02_german_credit.ipynb`) and
the diabetes data shipped with scikit-learn:

1. global summaries -- a few general rules describing every outcome of a tree; and
2. counterfactuals of random forests and extra-trees (experimental).

[gc]: https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data

In [1]:
import hashlib, os, urllib.request
import numpy as np

URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
SHA256 = 'b21f3d81db8071257d5ff1deaeba1fd4303b62712e6fcc9715c7a86202cb5871'
PATH = 'german.data'
if not os.path.isfile(PATH):
    urllib.request.urlretrieve(URL, PATH)
with open(PATH, 'rb') as f:
    assert hashlib.sha256(f.read()).hexdigest() == SHA256, 'unexpected data file'
rows = [line.split() for line in open(PATH) if line.strip()]
len(rows), rows[0]

(1000,
 ['A11',
  '6',
  'A34',
  'A43',
  '1169',
  'A65',
  'A75',
  '4',
  'A93',
  'A101',
  '4',
  'A121',
  '67',
  'A143',
  'A152',
  '2',
  'A173',
  '1',
  'A192',
  'A201',
  '1'])

In [2]:
FEATURES = [
    ('checking_account', {'A11': '< 0 DM', 'A12': '0 to 200 DM', 'A13': '>= 200 DM', 'A14': 'no checking account'}),
    ('duration', None),
    ('credit_history', {'A30': 'no credits taken', 'A31': 'all credits at this bank paid back',
                        'A32': 'existing credits paid back so far', 'A33': 'delays in the past',
                        'A34': 'critical account'}),
    ('purpose', {'A40': 'new car', 'A41': 'used car', 'A42': 'furniture', 'A43': 'radio/television',
                 'A44': 'domestic appliances', 'A45': 'repairs', 'A46': 'education', 'A47': 'vacation',
                 'A48': 'retraining', 'A49': 'business', 'A410': 'other'}),
    ('credit_amount', None),
    ('savings', {'A61': '< 100 DM', 'A62': '100 to 500 DM', 'A63': '500 to 1000 DM', 'A64': '>= 1000 DM',
                 'A65': 'unknown or none'}),
    ('employment_since', {'A71': 'unemployed', 'A72': '< 1 year', 'A73': '1 to 4 years', 'A74': '4 to 7 years',
                          'A75': '>= 7 years'}),
    ('installment_rate', None),
    ('personal_status_sex', {'A91': 'male, divorced/separated', 'A92': 'female, divorced/separated/married',
                             'A93': 'male, single', 'A94': 'male, married/widowed', 'A95': 'female, single'}),
    ('other_debtors', {'A101': 'none', 'A102': 'co-applicant', 'A103': 'guarantor'}),
    ('residence_since', None),
    ('property', {'A121': 'real estate', 'A122': 'savings agreement/life insurance', 'A123': 'car or other',
                  'A124': 'unknown or none'}),
    ('age', None),
    ('other_installment_plans', {'A141': 'bank', 'A142': 'stores', 'A143': 'none'}),
    ('housing', {'A151': 'rent', 'A152': 'own', 'A153': 'for free'}),
    ('existing_credits', None),
    ('job', {'A171': 'unemployed/unskilled non-resident', 'A172': 'unskilled resident',
             'A173': 'skilled employee', 'A174': 'management/self-employed'}),
    ('people_liable', None),
    ('telephone', {'A191': 'none', 'A192': 'yes'}),
    ('foreign_worker', {'A201': 'yes', 'A202': 'no'}),
]
y = np.array(['good' if r[-1] == '1' else 'bad' for r in rows])

In [3]:
from ctreex import Feature, FeatureSpec, TreeCounterfactualExplainer

columns = []
for j, (name, categories) in enumerate(FEATURES):
    if categories is None:
        columns.append((name, None))
    else:
        present = sorted(set(r[j] for r in rows), key=lambda c: int(c[1:]))
        columns += [(name, c) for c in present]
features = [Feature(name, 'numerical' if categories is None else 'categorical',
                    display_name=name.replace('_', ' '), value_names=categories or {},
                    integer=categories is None, ordered=name == 'employment_since',
                    values=list(categories) if name == 'employment_since' else [])
            for name, categories in FEATURES]
spec = FeatureSpec(columns, features=features, separator='=')
applicants = [{name: (int(r[j]) if categories is None else r[j])
               for j, (name, categories) in enumerate(FEATURES)} for r in rows]
X = np.stack([spec.encode(a) for a in applicants])
spec.fit_ranges(X)
print(spec.features['employment_since'].values, '| age:', spec.features['age'].minimum, 'to',
      spec.features['age'].maximum)

['A71', 'A72', 'A73', 'A74', 'A75'] | age: 19.0 to 75.0


In [4]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=42).fit(X, y)
explainer = TreeCounterfactualExplainer(clf, spec, class_names={'good': 'low risk', 'bad': 'high risk'})
i = int(np.flatnonzero(clf.predict(X) == 'bad')[0])
x = X[i]
explanation = explainer.explain_instance(x)
print('applicant', i, '| prediction:', explanation.prediction)
for distance, counterfactual in explanation.ranked()[:4]:
    print(distance, explainer.describe({0: counterfactual})[0])

applicant 1 | prediction: high risk
1 {'duration': {'is': [48], 'wannabe': ['less or equal to 22.5']}}
1 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}}
2 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}, 'other installment plans': {'is': ['none'], 'wannabe': ['bank', 'stores']}}
2 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}, 'duration': {'is': [48], 'wannabe': ['less or equal to 25.5']}}


## 1. Global summaries

`global_summary` describes every class with a few general rules:
every leaf of a class is grown into the largest rule -- one feature at a time -- whose data points all get that class,
and a greedy set cover keeps the rules needed to cover all the leaves of the class.
The rules are exact and may overlap;
every rule reports the leaves it contains, their training samples and their purity.

In [5]:
summary = explainer.global_summary()
for outcome, described in summary.items():
    print('{}: {} rules for {} leaves'.format(outcome, len(described.rules), len(described.leaves)))
    for rule in described.rules:
        print('   leaves {} | {} samples | purity {:.2f}'.format(rule.leaves, rule.samples, rule.purity))

high risk: 3 rules for 4 leaves
   leaves [12, 13] | 168 samples | purity 0.62
   leaves [10] | 42 samples | purity 0.55
   leaves [6] | 31 samples | purity 0.58
low risk: 3 rules for 9 leaves
   leaves [16, 17, 20, 21, 23, 24] | 394 samples | purity 0.88
   leaves [4, 5] | 318 samples | purity 0.69
   leaves [9] | 47 samples | purity 0.77


In [6]:
for sentence in explainer.summary_text(summary, integer_ranges=True):
    print(sentence)

IF checking account is < 0 DM, 0 to 200 DM or >= 200 DM AND savings is < 100 DM AND duration is at least 23 THEN the prediction is high risk.
IF checking account is < 0 DM, 0 to 200 DM or >= 200 DM AND credit history is existing credits paid back so far AND duration is at least 23 THEN the prediction is high risk.
IF checking account is < 0 DM, 0 to 200 DM or >= 200 DM AND property is unknown or none AND duration is at most 22 THEN the prediction is high risk.
IF checking account is no checking account THEN the prediction is low risk.
IF property is real estate, savings agreement/life insurance or car or other AND duration is at most 22 THEN the prediction is low risk.
IF credit history is no credits taken, all credits at this bank paid back, delays in the past or critical account AND savings is 100 to 500 DM, 500 to 1000 DM, >= 1000 DM or unknown or none AND duration is at least 23 THEN the prediction is low risk.


`max_rules` keeps the most general rules only;
the leaves they leave out are listed as `uncovered`:

In [7]:
for outcome, described in explainer.global_summary(max_rules=1).items():
    print(outcome, '| uncovered leaves:', described.uncovered)
    print('  ', explainer.summary_text({outcome: described})[0])

high risk | uncovered leaves: [6, 10]
   IF checking account is < 0 DM, 0 to 200 DM or >= 200 DM AND savings is < 100 DM AND duration is more than 22.5 THEN the prediction is high risk.
low risk | uncovered leaves: [4, 5, 9]
   IF checking account is no checking account THEN the prediction is low risk.


Regression trees are summarised by bands of predictions,
`(low, high)` meaning `low <= prediction < high`:

In [8]:
from sklearn.datasets import load_diabetes
from sklearn.tree import DecisionTreeRegressor

diabetes = load_diabetes(scaled=False)
DISPLAY = {'bmi': 'body mass index',
           'bp': 'blood pressure',
           's1': 'total cholesterol',
           's2': 'LDL',
           's3': 'HDL',
           's4': 'cholesterol / HDL',
           's5': 'log triglycerides',
           's6': 'blood sugar'}
diabetes_spec = FeatureSpec([(name, None) for name in diabetes.feature_names],
                            features=[Feature(name, 'numerical', display_name=DISPLAY.get(name, name))
                                      for name in diabetes.feature_names])
reg = DecisionTreeRegressor(max_depth=4, min_samples_leaf=20, random_state=0).fit(diabetes.data, diabetes.target)
reg_explainer = TreeCounterfactualExplainer(reg, diabetes_spec)
print(reg_explainer.task, '|', reg.get_n_leaves(), 'leaves')

patient = diabetes.data[65]
print('prediction:', reg_explainer.tree.predict(patient.reshape(1, -1))[0])

regression | 13 leaves
prediction: 150.35


In [9]:
bands = [(None, 100), (100, 200), (200, None)]
regression_summary = reg_explainer.global_summary(bands=bands)
for band, described in regression_summary.items():
    print(band, '|', len(described.rules), 'rules for', len(described.leaves), 'leaves')
for sentence in reg_explainer.summary_text({b: d for b, d in regression_summary.items() if b != (100, 200)}):
    print(sentence)

(None, 100) | 2 rules for 3 leaves
(100, 200) | 5 rules for 7 leaves
(200, None) | 3 rules for 3 leaves
IF body mass index is less or equal to 26.949999 AND HDL is more than 55.5 AND log triglycerides is less or equal to 4.6001496 THEN the prediction is less than 100.
IF body mass index is less or equal to 26.949999 AND log triglycerides is less or equal to 4.16665 THEN the prediction is less than 100.
IF body mass index is more than 27.75 AND blood pressure is more than 99.5 AND log triglycerides is more than 4.6001496 THEN the prediction is at least 200.
IF body mass index is more than 32.75 AND log triglycerides is more than 4.6001496 THEN the prediction is at least 200.
IF body mass index is between 24.349998 and 27.75 AND HDL is less or equal to 37.5 AND log triglycerides is more than 4.6001496 THEN the prediction is at least 200.


## 2. Tree ensembles (experimental)

`EnsembleCounterfactualExplainer` explains random forests and extra-trees.
It describes the feature space with the cells between the thresholds of all the trees
and evaluates the ensemble on every cell reachable by changing one or two features,
so the counterfactuals at distances 1 and 2 are exact.

In [10]:
from sklearn.ensemble import RandomForestClassifier
from ctreex.ensemble import EnsembleCounterfactualExplainer

forest = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42).fit(X, y)
forest_explainer = EnsembleCounterfactualExplainer(forest, spec, class_names={'good': 'low risk', 'bad': 'high risk'})
print(len(forest_explainer.meta_features), 'cells | accuracy', (forest.predict(X) == y).mean())

j = int(np.flatnonzero(forest.predict(X) == 'bad')[0])
forest_explanation = forest_explainer.explain_instance(X[j])
print('applicant', j, '| prediction:', forest_explanation.prediction)
for distance in forest_explanation.distances:
    print('distance', distance, ':', len(forest_explanation.at_distance(distance).ranked), 'counterfactuals')
for sentence in forest_explainer.explanation_text(forest_explanation, n=5, integer_ranges=True):
    print(sentence)

230 cells | accuracy 0.742


applicant 11 | prediction: high risk
distance 1 : 3 counterfactuals
distance 2 : 49 counterfactuals
The prediction is high risk.
Had duration been at most 34 (instead of 48), the prediction would have been low risk.
Had savings been 100 to 500 DM, 500 to 1000 DM, >= 1000 DM or unknown or none (instead of < 100 DM), the prediction would have been low risk.
Had checking account been >= 200 DM or no checking account (instead of < 0 DM), the prediction would have been low risk.
Had age been at least 28 (instead of 24) and duration been between 35 and 43 (instead of 48), the prediction would have been low risk.
Had checking account been 0 to 200 DM (instead of < 0 DM) and credit amount been at most 3918 (instead of 4308), the prediction would have been low risk.


The single tree explains the same applicant differently:

In [11]:
for sentence in explainer.explanation_text(explainer.explain_instance(X[j]), n=3, integer_ranges=True):
    print(sentence)

The prediction is high risk.
Had duration been at most 22 (instead of 48), the prediction would have been low risk.
Had checking account been no checking account (instead of < 0 DM), the prediction would have been low risk.
Had checking account been no checking account (instead of < 0 DM) and other installment plans been bank or stores (instead of none), the prediction would have been low risk.


At distance 2, only the pairs of changes that are both needed are reported
(neither change alone reaches the target);
the data points of a counterfactual get its outcome:

In [12]:
for distance, classes, counterfactual in forest_explanation.ranked(with_classes=True):
    if distance == 2:
        print(forest_explainer.describe({0: counterfactual}, integer_ranges=True)[0])
        break
samples = forest_explainer.sample_counterfactual(X[j], counterfactual, n=5, random_state=0)
print('predictions of the samples:', sorted(set(forest_explainer.predict(samples).tolist())))

{'age': {'is': [24], 'wannabe': ['at least 28']}, 'duration': {'is': [48], 'wannabe': ['between 35 and 43']}}
predictions of the samples: ['low risk']


The number of evaluated points grows with the square of the number of cells,
so `max_evaluations` limits it and `included` restricts the features to change.
Ensembles are stored as JSON with `EnsembleStructure`.

In [13]:
import json

from ctreex.ensemble import EnsembleStructure, load_ensemble

limited = EnsembleCounterfactualExplainer(forest, spec, max_evaluations=2000)
try:
    limited.explain_instance(X[j])
except ValueError as error:
    print(error)
print(len(limited.explain_instance(X[j], included=['duration', 'age']).ranked()),
      'counterfactuals changing duration or age')

stored = load_ensemble(json.loads(EnsembleStructure.from_sklearn(forest).to_json()))
print(stored, '| same predictions:', (stored.predict(X) == forest.predict(X)).all())

The explanation needs 16946 evaluations (more than max_evaluations=2000); restrict the features with included or explain at max_distance=1.
3 counterfactuals changing duration or age
EnsembleStructure(n_trees=100, n_features=61, classes=['bad', 'good']) | same predictions: True


Regression ensembles need a target:

In [14]:
from sklearn.ensemble import ExtraTreesRegressor
from ctreex import ValueChange

trees = ExtraTreesRegressor(n_estimators=20, max_depth=4, random_state=0).fit(diabetes.data, diabetes.target)
trees_explainer = EnsembleCounterfactualExplainer(trees, diabetes_spec)
trees_explanation = trees_explainer.explain_instance(patient, target=ValueChange(+50), max_distance=1)
for sentence in trees_explainer.explanation_text(trees_explanation, n=3):
    print(sentence)

The prediction is 169.366.
Had body mass index been between 33.445698 and 34.27138 (instead of 24.1), the prediction would have been 237.261 (instead of 169.366).
Had body mass index been between 35.064114 and 37.544807 (instead of 24.1), the prediction would have been 249.064 (instead of 169.366).
Had body mass index been between 33.1123 and 33.445698 (instead of 24.1), the prediction would have been 232.837 (instead of 169.366).
